In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class WESADDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [2]:
DATA_DIR = "../data/processed_ds"
SUBJECTS = [s for s in range(2, 18) if s != 12]  # S2-S17, excluding S12

# Fixed validation subject for the entire study.
# Chosen once, used in every fold (except the one where it's the test subject).
FIXED_VAL_SUBJECT = 17
# Secondary fixed validation subject — used only for the fold where
# test_sid == FIXED_VAL_SUBJECT (S17 can't validate on itself).
SECONDARY_VAL_SUBJECT = 16

def load_subject(sid):
    X = np.load(f"{DATA_DIR}/S{sid}_X.npy")
    y = np.load(f"{DATA_DIR}/S{sid}_y.npy")
    return X, y

def get_val_subject(test_sid):
    """Returns the validation subject for a given test fold."""
    if test_sid == FIXED_VAL_SUBJECT:
        return SECONDARY_VAL_SUBJECT
    return FIXED_VAL_SUBJECT

def load_all_except(test_sid, val_sid):
    Xs, ys = [], []
    for sid in SUBJECTS:
        if sid == test_sid or sid == val_sid:
            continue
        X, y = load_subject(sid)
        Xs.append(X)
        ys.append(y)
    return np.concatenate(Xs), np.concatenate(ys)

In [3]:
import torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
import copy

def train_one_fold(model, train_loader, val_loader, device, epochs=50, patience=10, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # Class weights to handle imbalance
    all_y = torch.cat([y for _, y in train_loader])
    class_counts = torch.bincount(all_y)
    class_weights = (1.0 / class_counts.float())
    class_weights = class_weights / class_weights.sum() * len(class_counts)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    best_val_f1 = -1
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        scheduler.step()

        # --- Validate ---
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb = Xb.to(device)
                out = model(Xb)
                preds = out.argmax(dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(yb.numpy())

        val_f1 = f1_score(val_true, val_preds, average='macro', zero_division=0)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch+1}/{epochs} | train_loss={train_loss/len(train_loader):.4f} | val_f1={val_f1:.4f}")

    model.load_state_dict(best_state)
    return model, best_val_f1

In [6]:
import sys
import os

sys.path.append(os.path.abspath("..")) # ensures notebooks/models/ and notebooks/experiments/ are importable
from models.cnn_encoder import MultiModalCNN
from torch.utils.data import DataLoader

device = torch.device("cpu")
torch.manual_seed(42)

# Pick test subject S2; validation subject comes from the fixed assignment
test_sid = 2
val_sid = get_val_subject(test_sid)  # = 17

train_X, train_y = load_all_except(test_sid, val_sid)
val_X, val_y = load_subject(val_sid)
test_X, test_y = load_subject(test_sid)

train_ds = WESADDataset(train_X, train_y)
val_ds = WESADDataset(val_X, val_y)
test_ds = WESADDataset(test_X, test_y)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

model = MultiModalCNN().to(device)

import time
start = time.time()
model, best_f1 = train_one_fold(model, train_loader, val_loader, device, epochs=10)  # short run first
print(f"Time for 10 epochs: {time.time()-start:.1f}s")
print(f"Best val F1: {best_f1:.4f}")

  Epoch 1/10 | train_loss=0.3795 | val_f1=0.8975
  Epoch 6/10 | train_loss=0.0969 | val_f1=0.9923
  Epoch 10/10 | train_loss=0.0281 | val_f1=0.9923
Time for 10 epochs: 250.0s
Best val F1: 0.9923


In [7]:
print(train_X.shape)

(1897, 4, 3000)


In [ ]:
import sys
import torch
from torch.utils.data import DataLoader
import numpy as np
import time

sys.path.append("..")

from models.cnn_encoder import MultiModalCNN

device = torch.device("cpu")

torch.set_num_threads(12)
torch.manual_seed(42)
np.random.seed(42)

test_sid = 2
val_sid = get_val_subject(test_sid)

train_X, train_y = load_all_except(test_sid, val_sid)
val_X, val_y = load_subject(val_sid)

print("Train shape:", train_X.shape)
print("Val shape:", val_X.shape)

train_loader = DataLoader(
    WESADDataset(train_X, train_y),
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    WESADDataset(val_X, val_y),
    batch_size=32,
    shuffle=False
)

# Explicitly test reduced model
model = MultiModalCNN(
    embed_dim=128,
    n_modalities=4,
    n_classes=2
).to(device)

start = time.time()

model, best_f1 = train_one_fold(
    model,
    train_loader,
    val_loader,
    device,
    epochs=10,
    patience=10
)

elapsed = time.time() - start

print("\n=== 128-dim Benchmark ===")
print(f"Time for 10 epochs: {elapsed:.1f}s")
print(f"Best val F1: {best_f1:.4f}")

Train shape: (1897, 4, 3000)
Val shape: (150, 4, 3000)
  Epoch 1/10 | train_loss=0.3925 | val_f1=0.9335
  Epoch 6/10 | train_loss=0.0605 | val_f1=0.9923
  Epoch 10/10 | train_loss=0.0389 | val_f1=0.9923

=== 128-dim Benchmark ===
Time for 10 epochs: 150.6s
Best val F1: 1.0000
